# 3 · Production serving + Locust: vLLM (AWQ) + BentoML + Qdrant, 50 concurrent users

**Kaggle · GPU T4 x2 · Internet ON.** Stack under test: **BentoML** service (retrieval + rerank + prompt) -> **vLLM** serving `Qwen3-8B-AWQ`, Qdrant as vector store.
Produces: the streaming `curl -N` proof, `reports/locust_u1.*`, `reports/locust_u50.*` (+ HTML), `reports/load_test.md` with p50/p95/p99. Budget ~45 GPU-minutes.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/AhmedYasser06/Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## Corpus
`dvc pull` fetches the PDF, the JSON corpus, the chunks and the embeddings from the shared DVC remote.
If the remote needs credentials, add Kaggle secrets `DVC_USER` / `DVC_PASSWORD` (Add-ons -> Secrets).
Fallback: attach a Kaggle dataset containing your `data/processed/` folder.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
try:
    s = UserSecretsClient()
    user, pw = s.get_secret("DVC_USER"), s.get_secret("DVC_PASSWORD")
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
    !dvc remote modify --local origin auth basic
    !dvc remote modify --local origin user {user}
    !dvc remote modify --local origin password {pw}
except Exception as e:
    print("no DVC secrets, trying anonymous pull:", e)
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
!dvc pull
!ls -la data/processed
# fallback if dvc pull failed:  !mkdir -p data/processed && cp /kaggle/input/*/final-*.json /kaggle/input/*/embedded-*.json data/processed/

## vLLM in its own environment
vLLM pins its own torch/transformers, so it gets a separate venv (made with `uv`: Kaggle's `python -m venv` has no ensurepip). Kaggle doesn't allow `!cmd &`, so servers start with `subprocess.Popen`.

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
import os, signal, subprocess, time, urllib.request

def start(cmd, log_path, env=None):
    return subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def wait_ready(url, log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen(url, timeout=5); print(f"ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- log:"); os.system(f"tail -n 30 {log_path}"); return False

def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    try: proc.wait(timeout=120)
    except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(10)

def start_vllm(model, log_path, util=0.75):
    return start(["/kaggle/vllm-env/bin/vllm", "serve", model, "--tensor-parallel-size", "2",
                  "--dtype", "half", "--max-model-len", "8192", "--gpu-memory-utilization", str(util),
                  "--port", "8000"], log_path, env={"PYTHONPATH": ""})

In [ ]:
# API environment (BentoML + Locust) in its own venv so it never fights vLLM
!uv venv /kaggle/api-env --python 3.11
!uv pip install -q --python /kaggle/api-env/bin/python -r requirements-serving.txt
!uv pip install -q --python /kaggle/api-env/bin/python -e . --no-deps
!mkdir -p eval_out reports

## Qdrant server + index

In [ ]:
!curl -sSL -o /kaggle/working/qdrant.tgz https://github.com/qdrant/qdrant/releases/download/v1.19.0/qdrant-x86_64-unknown-linux-musl.tar.gz
!tar xzf /kaggle/working/qdrant.tgz -C /kaggle/working
qdrant = start(["/kaggle/working/qdrant"], "/kaggle/working/qdrant.log",
               env={"QDRANT__STORAGE__STORAGE_PATH": "/kaggle/working/qdrant_storage", "QDRANT__TELEMETRY_DISABLED": "true"})
wait_ready("http://localhost:6333/", "/kaggle/working/qdrant.log", minutes=2)
!QDRANT_URL=http://localhost:6333 /kaggle/api-env/bin/python -m src.scripts.ensure_index

## vLLM (AWQ) and the BentoML service

In [ ]:
vllm = start_vllm("Qwen/Qwen3-8B-AWQ", "/kaggle/working/vllm.log", util=0.6)
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm.log")
!grep "Model loading took" /kaggle/working/vllm.log | head -1

In [ ]:
bento = start(["/kaggle/api-env/bin/bentoml", "serve", "src.serving.bento_service:LegalRAG", "--port", "3000"],
              "/kaggle/working/bento.log",
              env={"LLM_PROVIDER": "VLLM", "VLLM_BASE_URL": "http://localhost:8000/v1", "VLLM_MODEL_ID": "Qwen/Qwen3-8B-AWQ",
                   "QDRANT_URL": "http://localhost:6333", "RERANKER_DEVICE": "cuda:0", "EMBEDDING_DEVICE": "cuda:0", "PYTHONPATH": "/kaggle/working/repo"})
wait_ready("http://localhost:3000/readyz", "/kaggle/working/bento.log", minutes=15)

### Streaming proof (tokens appear progressively; keep this output for the README)

In [ ]:
!curl -N -s -X POST http://localhost:3000/ask -H 'Content-Type: application/json' -d '{"question": "ما هي شروط صحة العقد؟"}' | tee reports/streaming_demo.txt

## Locust: 1 user (baseline) then 50 users x 5 min against `/ask_json`

In [ ]:
env = "ENDPOINT=/ask_json QUESTIONS=data/evaluation/rag_eval.json"
!{env} /kaggle/api-env/bin/locust -f load_test/locustfile.py --host http://localhost:3000 --headless -u 1 -r 1 -t 2m --csv reports/locust_u1 --html reports/locust_u1.html --only-summary > eval_out/locust_u1.log 2>&1; tail -n 5 eval_out/locust_u1.log

In [ ]:
# Locust may crash at shutdown (ValueError: I/O operation on closed file); the CSV is still complete.
# If this cell is still running after ~8 min, interrupt it.
!{env} /kaggle/api-env/bin/locust -f load_test/locustfile.py --host http://localhost:3000 --headless -u 50 -r 5 -t 5m --csv reports/locust_u50 --html reports/locust_u50.html --only-summary > eval_out/locust_u50.log 2>&1; tail -n 5 eval_out/locust_u50.log

In [ ]:
# streaming run: time-to-first-token under 20 users, 1 minute
!ENDPOINT=/ask STREAM=1 QUESTIONS=data/evaluation/rag_eval.json /kaggle/api-env/bin/locust -f load_test/locustfile.py --host http://localhost:3000 --headless -u 20 -r 5 -t 1m --csv reports/locust_stream_u20 --only-summary > eval_out/locust_stream.log 2>&1; tail -n 3 eval_out/locust_stream.log
!python scripts/summarize_locust.py reports/locust_u1 reports/locust_u50 reports/locust_stream_u20 --out reports/load_test.md

In [ ]:
!cp /kaggle/working/vllm.log /kaggle/working/bento.log eval_out/
!zip -qr /kaggle/working/load_test_results.zip reports/locust_* reports/load_test.md reports/streaming_demo.txt eval_out/*.log
from IPython.display import FileLink
display(FileLink("/kaggle/working/load_test_results.zip"))